# Direct Postgres Connection via Spark JDBC
This notebook demonstrates how to connect directly to the Postgres database (Hive Metastore) using Spark and the JDBC driver. This allows you to query the metadata (databases, tables, columns) directly from the underlying relational storage.

In [1]:
from pyspark.sql import SparkSession
import os

# Note: We include the postgres JDBC driver in the extraClassPath
# The IP address '172.19.0.2' is the internal Docker IP for the Postgres container
# Update this if your container IP changes.

# We use spark.jars.packages to automatically download and handle the driver
# Or we can specify a local path if we have it downloaded.
spark = SparkSession.builder \
    .appName("PostgresDirectAccess") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

print("Spark Session created with Postgres JDBC driver.")

26/04/22 03:21:31 WARN Utils: Your hostname, KAOs-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.18.9 instead (on interface en0)
26/04/22 03:21:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/kaoleangseng/.ivy2/cache
The jars for the packages stored in: /Users/kaoleangseng/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1cada692-7f32-47fc-977d-cb8f2afafeea;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.3/postgresql-42.7.3.jar ...
	[SUCCESSFUL ] org.postgresql#postgresql;42.7.3!postgresql.jar (808ms)
downloading https://repo1.maven.org/maven2/org/checkerframework/checker-qual/3.42.0/checker-qual-3.42.0.jar ...
	[SUCCESSFUL ] org.checkerframework#checker-qual;3.42.0!checker-qual.jar (210ms)
:: resolution report :: resolve 1734ms :: artifacts dl 1032ms
	:: modules in use:
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	--------------

Spark Session created with Postgres JDBC driver.


## Connection Configuration
We define the JDBC URL, properties, and credentials.

In [2]:
# JDBC Connection Details
jdbc_url = "jdbc:postgresql://172.19.0.2:5432/metastore"
connection_properties = {
    "user": "hive",
    "password": "password",
    "driver": "org.postgresql.Driver"
}

# Test connection by listing tables in the Hive Metastore
try:
    # 'TBLS' is the standard Hive Metastore table that stores table names
    df_tables = spark.read.jdbc(url=jdbc_url, table="\"TBLS\"", properties=connection_properties)
    print("Successfully connected to Postgres!")
    df_tables.show(5)
except Exception as e:
    print(f"Error connecting: {e}")

Error connecting: An error occurred while calling o30.jdbc.
: org.postgresql.util.PSQLException: The connection attempt failed.
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:358)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:54)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:273)
	at org.postgresql.Driver.makeConnection(Driver.java:446)
	at org.postgresql.Driver.connect(Driver.java:298)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.scala:160)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1$adapted(JdbcDialects.scala:156)
	at org.apache.spark.sql.execution.datasourc

## Querying Specific Metadata
You can use SQL queries to explore the Postgres schema. For example, let's see all the tables registered in Hive.

In [3]:
# Querying specific columns from the 'TBLS' (Tables) and 'DBS' (Databases) tables
query = """
(SELECT 
    d. "NAME" as database_name, 
    t. "TBL_NAME" as table_name, 
    t. "OWNER" as owner 
 FROM "TBLS" t 
 JOIN "DBS" d ON t."DB_ID" = d."DB_ID") as table_info
"""

df_info = spark.read.jdbc(url=jdbc_url, table=query, properties=connection_properties)
df_info.show()

Py4JJavaError: An error occurred while calling o34.jdbc.
: org.postgresql.util.PSQLException: The connection attempt failed.
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:358)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:54)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:273)
	at org.postgresql.Driver.makeConnection(Driver.java:446)
	at org.postgresql.Driver.connect(Driver.java:298)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.scala:160)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1$adapted(JdbcDialects.scala:156)
	at org.apache.spark.sql.execution.datasources.